# CsvUploader — component test

Demonstrates:
1. Single-file upload (basic)
2. Multi-file upload
3. Custom client-side parser injection via `assets/csv_parser.js`
4. Server-side DataFrame reconstruction with **PyArrow** (zero-copy, memory-efficient)
5. All three `variant` modes: `full`, `auto-upload`, `download-only`

## Data format

On submit the component sets `contents`, a list of column-major objects:
```json
{
  "name": "data.csv",
  "columns": ["ts", "power_mw", "site"],
  "data": [["2024-01-01", "2024-01-02"], [12.5, 14.1], ["A", "A"]],
  "n_rows": 2,
  "n_cols": 3
}
```
Reconstruct as a DataFrame: `pa.table(dict(zip(d["columns"], d["data"]))).to_pandas()`

In [ ]:
import os
import textwrap
import pyarrow as pa
import pandas as pd
from dash import Dash, Input, Output, State, html
import dash_material_components as dmc

## 1 · Custom parser

Write a JavaScript file into the Dash `assets/` folder.
Dash automatically serves every file in that folder, so the script is injected
into the page before the component renders.

The parser function receives `(rows, fields)` where:
- `rows` — array of row objects `{column: value, ...}` (numbers already cast by PapaParse)
- `fields` — ordered array of column names

It must return either `{success: true, data: rows, fields}` or
`{success: false, errorMessage: "reason"}`.

In [ ]:
os.makedirs('assets', exist_ok=True)

# Parser for an energy time-series CSV:
#   required columns: timestamp, power_mw
#   optional:         site_id
js = textwrap.dedent("""
    window.DashCsvParsers = window.DashCsvParsers || {};

    window.DashCsvParsers["energy_ts"] = function (rows, fields) {
        var required = ["timestamp", "power_mw"];

        // Schema check
        for (var i = 0; i < required.length; i++) {
            if (!fields.includes(required[i]))
                return { success: false, errorMessage: "Missing column: " + required[i] };
        }

        // Value check: power_mw must be numeric
        for (var j = 0; j < rows.length; j++) {
            var val = rows[j]["power_mw"];
            if (typeof val !== "number" || isNaN(val))
                return { success: false, errorMessage: "power_mw must be numeric (row " + (j + 1) + ")" };
            rows[j]["power_mw"] = 0.0;
        }

        return { success: true, data: rows, fields: fields };
    };
""").strip()

with open('assets/csv_parser.js', 'w') as f:
    f.write(js)

print('Parser written to assets/csv_parser.js')

## 2 · Sample CSV files

Create two small CSV files for manual testing.

In [ ]:
good_csv = textwrap.dedent("""
    timestamp,power_mw,site_id
    2024-01-01T00:00:00,12.5,SITE_A
    2024-01-01T01:00:00,14.1,SITE_A
    2024-01-01T02:00:00,11.8,SITE_A
    2024-01-01T03:00:00,9.3,SITE_A
""").strip()

bad_csv = textwrap.dedent("""
    date,value
    2024-01-01,100
    2024-01-02,200
""").strip()

with open('/tmp/energy_good.csv', 'w') as f:
    f.write(good_csv)

with open('/tmp/energy_bad.csv', 'w') as f:
    f.write(bad_csv)

print('Sample CSVs written to /tmp/')

## 3 · PyArrow helper

The component sends column-major data.  
PyArrow reconstructs a typed table with a single pass — no re-parsing, minimal allocations.

In [ ]:
def to_dataframe(file_data: dict) -> pd.DataFrame:
    """Convert a CsvUploader FileOutput dict to a pandas DataFrame via PyArrow.

    PyArrow infers the optimal column dtype from the already-typed values
    (numbers were cast client-side by PapaParse), so no extra parsing step
    is needed server-side.
    """
    col_arrays = dict(zip(file_data["columns"], file_data["data"]))
    return pa.table(col_arrays).to_pandas()


# Quick smoke-test with synthetic data
synthetic = {
    "name": "test.csv",
    "columns": ["timestamp", "power_mw", "site_id"],
    "data": [
        ["2024-01-01T00:00:00", "2024-01-01T01:00:00"],
        [12.5, 14.1],
        ["SITE_A", "SITE_A"],
    ],
    "n_rows": 2,
    "n_cols": 3,
}
df_test = to_dataframe(synthetic)
print(df_test.dtypes)
df_test

## 4 · Enedis parser

`assets/enedis.js` registers `window.DashCsvParsers["enedis"]`.
Dash auto-serves every file in `assets/`, so the parser is injected before
any uploader renders — no explicit `<script>` tag needed.

The parser source lives in `src/parsers/enedis.js` and is copied to `assets/`
by the setup cell below — no need to maintain a separate copy.

The parser auto-detects all four Enedis export templates:
| Template | Key columns | Timestamp |
|----------|------------|----------|
| **Simple** | 2 columns | tz-aware |
| **M021** | `Identifiant PRM` (multi-header) | tz-aware ISO |
| **M022** | `PRM` + French date/time | naive Europe/Paris |
| **M023** | `Horodate` + `Valeur` (W rows only) | naive Europe/Paris |

Output fields: `delivery_from` (UTC ISO), `load_value` (MW), `meterpoint_id`.

`meterpoint_id` is constant within a file, so it is written only on the first
row to minimise transfer size; all subsequent rows carry `null`.
Recover server-side with:
```python
meterpoint_id = df["meterpoint_id"].iloc[0]   # scalar
df = df.drop(columns="meterpoint_id")          # or ffill() if you need it on every row
```

In [ ]:
# template files for testing can be found in src/tests/fixtures

## 5 · Dash application

Seven uploaders across two tab groups:

**Parsers & file-count**
- **Single** — no parser, one file at a time
- **Multi** — no parser, multiple files
- **Validated** — uses the `energy_ts` parser; client-side schema check
- **Enedis** — uses the `enedis` parser; auto-detects all four Enedis templates

**Variants**
- **full** — default; explicit Upload button sends data on click
- **auto-upload** — data pushed to Dash automatically after each successful parse; no button
- **download-only** — parse and download locally; never uploads to server

In [ ]:
# ── Uploaders ─────────────────────────────────────────────────────────────

uploader_single = dmc.CsvUploader(
    id="uploader-single",
    labelText="Single CSV file",
    multiple=False,
    maxSizeMb=10,
)

uploader_multi = dmc.CsvUploader(
    id="uploader-multi",
    labelText="Multiple CSV files",
    multiple=True,
    maxSizeMb=10,
)

uploader_validated = dmc.CsvUploader(
    id="uploader-validated",
    labelText="Energy time-series (validated)",
    multiple=True,
    maxSizeMb=10,
    # references window.DashCsvParsers["energy_ts"] from assets/csv_parser.js
    parserId="energy_ts",
)

uploader_enedis = dmc.CsvUploader(
    id="uploader-enedis",
    labelText="Enedis CSV (auto-detects Simple / M021 / M022 / M023)",
    multiple=True,
    maxSizeMb=50,
    # references window.DashCsvParsers["enedis"] from assets/enedis.js
    parserId="enedis",
)

# variant="full" is the default — explicit Upload button
uploader_full = dmc.CsvUploader(
    id="uploader-full",
    labelText="full — click Upload to send",
    multiple=True,
    maxSizeMb=10,
    variant="full",
)

# variant="auto-upload" — data pushed to Dash as soon as each file parses successfully
uploader_auto = dmc.CsvUploader(
    id="uploader-auto",
    labelText="auto-upload — data sent automatically after each parse",
    multiple=True,
    maxSizeMb=10,
    variant="auto-upload",
)

# variant="download-only" — no upload; user can only download parsed files
uploader_dl = dmc.CsvUploader(
    id="uploader-dl",
    labelText="download-only — parse locally, never uploads",
    multiple=True,
    maxSizeMb=10,
    variant="download-only",
)

# ── Output displays ────────────────────────────────────────────────────────

output_single    = dmc.Typography(id="output-single",    text="No data yet.")
output_multi     = dmc.Typography(id="output-multi",     text="No data yet.")
output_validated = dmc.Typography(id="output-validated", text="No data yet.")
output_enedis    = dmc.Typography(id="output-enedis",    text="No data yet.")
output_full      = dmc.Typography(id="output-full",      text="No data yet.")
output_auto      = dmc.Typography(id="output-auto",      text="No data yet.")

# ── Layout ─────────────────────────────────────────────────────────────────

tab = dmc.Tab(
    children=[
        dmc.Box(children=[uploader_single,    output_single]),
        dmc.Box(children=[uploader_multi,     output_multi]),
        dmc.Box(children=[uploader_validated, output_validated]),
        dmc.Box(children=[uploader_enedis,    output_enedis]),
        dmc.Box(children=[uploader_full,      output_full]),
        dmc.Box(children=[uploader_auto,      output_auto]),
        dmc.Box(children=[uploader_dl]),
    ],
    tabs=[
        {"label": "Single"},
        {"label": "Multi"},
        {"label": "Validated (energy_ts)"},
        {"label": "Enedis"},
        {"label": "Variant: full"},
        {"label": "Variant: auto-upload"},
        {"label": "Variant: download-only"},
    ],
)

section = dmc.Section(
    orientation="columns",
    children=tab,
    cards=[{"title": "CsvUploader"}],
)
page   = dmc.Page(orientation="columns", children=section)
navbar = dmc.NavBar(title="CsvUploader test")
layout = dmc.Dashboard(children=[navbar, page])

In [ ]:
app = Dash(__name__)
app.layout = layout


def summarise_files(contents: list) -> str:
    """Build a human-readable summary and reconstruct DataFrames via PyArrow."""
    if not contents:
        return "No data submitted yet."

    parts = []
    for d in contents:
        df = to_dataframe(d)
        parts.append(
            f"{d['name']}  →  {d['n_rows']} rows × {d['n_cols']} cols\n"
            f"dtypes: {dict(df.dtypes.astype(str))}\n"
            f"{df.head(3).to_string(index=False)}"
        )
    return "\n\n".join(parts)


@app.callback(
    Output("output-single", "text"),
    Input("uploader-single", "nSubmits"),
    State("uploader-single", "contents"),
    prevent_initial_call=True,
)
def on_single_submit(n_submits, contents):
    return summarise_files(contents or [])


@app.callback(
    Output("output-multi", "text"),
    Input("uploader-multi", "nSubmits"),
    State("uploader-multi", "contents"),
    prevent_initial_call=True,
)
def on_multi_submit(n_submits, contents):
    return summarise_files(contents or [])


@app.callback(
    Output("output-validated", "text"),
    Input("uploader-validated", "nSubmits"),
    State("uploader-validated", "contents"),
    prevent_initial_call=True,
)
def on_validated_submit(n_submits, contents):
    return summarise_files(contents or [])


@app.callback(
    Output("output-enedis", "text"),
    Input("uploader-enedis", "nSubmits"),
    State("uploader-enedis", "contents"),
    prevent_initial_call=True,
)
def on_enedis_submit(n_submits, contents):
    return summarise_files(contents or [])


# variant="full" — same pattern as the standard uploaders above
@app.callback(
    Output("output-full", "text"),
    Input("uploader-full", "nSubmits"),
    State("uploader-full", "contents"),
    prevent_initial_call=True,
)
def on_full_submit(n_submits, contents):
    return summarise_files(contents or [])


# variant="auto-upload" — listen to `contents` directly; fires on every successful parse
@app.callback(
    Output("output-auto", "text"),
    Input("uploader-auto", "contents"),
    prevent_initial_call=True,
)
def on_auto_upload(contents):
    return summarise_files(contents or [])


# variant="download-only" — no server callback; data never leaves the browser


app.run_server(mode="inline", host="0.0.0.0", port=8010, debug=True)